# B2-024-gpu-scientific-ml-capstone — Practice p04 — Solution

**Type:** mc · **Difficulty:** core · **Concepts:** semi-supervised-pseudo-labeling

*20 minutes.*  
**Set:** A  
**Program layer:** Round 2 extension  
**Compute:** `compute.policy: cpu` · seed `20261022`  
**Qualified prerequisites:** `book1:F1-scientific-python`, `book1:F3-matrices`, `book1:F4-multivar-calculus`, `book1:F5-probability`, `book1:C1-ml-fundamentals`, `book1:C2-linear-models`, `book1:C3-gradient-descent`, `book1:C5-neural-networks`, `book1:C6-pytorch`, `book1:C7-cnn-transfer`, `book1:C10-competition-craft`, `book1:C11-neural-training`, `book1:C12-classical-models`, `B2-023-generative-models-diffusion`  
**Remediation links actually used:** [book1:F1-scientific-python](../../../../book1/units/F1-scientific-python/lesson.ipynb), [book1:F3-matrices](../../../../book1/units/F3-matrices/lesson.ipynb), [book1:F4-multivar-calculus](../../../../book1/units/F4-multivar-calculus/lesson.ipynb), [book1:F5-probability](../../../../book1/units/F5-probability/lesson.ipynb), [book1:C1-ml-fundamentals](../../../../book1/units/C1-ml-fundamentals/lesson.ipynb), [book1:C2-linear-models](../../../../book1/units/C2-linear-models/lesson.ipynb), [book1:C3-gradient-descent](../../../../book1/units/C3-gradient-descent/lesson.ipynb), [book1:C5-neural-networks](../../../../book1/units/C5-neural-networks/lesson.ipynb), [book1:C6-pytorch](../../../../book1/units/C6-pytorch/lesson.ipynb), [book1:C7-cnn-transfer](../../../../book1/units/C7-cnn-transfer/lesson.ipynb), [book1:C10-competition-craft](../../../../book1/units/C10-competition-craft/lesson.ipynb), [book1:C11-neural-training](../../../../book1/units/C11-neural-training/lesson.ipynb), [book1:C12-classical-models](../../../../book1/units/C12-classical-models/lesson.ipynb), [B2-023-generative-models-diffusion](../../B2-023-generative-models-diffusion/lesson.ipynb).

## Solution

**Step 1 — predicted label and confidence of each row** (arg-max and max of the row):

| index | label | confidence | candidate ($\ge 0.80$)? |
|---:|---:|---:|---|
| 0 | 0 | 0.91 | yes |
| 1 | 1 | 0.85 | yes |
| 2 | 0 | 0.82 | yes |
| 3 | 2 | 0.90 | yes |
| 4 | 0 | 0.95 | yes |
| 5 | 1 | 0.65 | no |
| 6 | 0 | 0.80 | yes (equality counts: the comparison is `>=`) |
| 7 | 2 | 0.80 | yes (equality counts) |

**Step 2 — candidates per predicted class, highest confidence first:**

* class 0: index 4 (0.95), index 0 (0.91), index 2 (0.82), index 6 (0.80);
* class 1: index 1 (0.85);
* class 2: index 3 (0.90), index 7 (0.80).

**Step 3 — cap of 2 per class:** class 0 keeps 4 and 0 (drops 2 and 6); class 1 keeps 1; class 2 keeps 3 and 7.
No confidence ties occur inside a class, so the lower-index tie rule is not needed here.

**Step 4 — sort ascending:** indices `[0, 1, 3, 4, 7]`, labels `[0, 1, 2, 0, 2]`.

**Answer: B.**

**Rules broken by the wrong options.**

* **A** drops index 7: it uses `>` instead of `>=` at the threshold (0.80 is not strictly greater than 0.80).
* **C** keeps four class-0 images (0, 2, 4, 6): it ignores the per-class cap.
* **D** keeps index 2 (0.82) instead of index 4 (0.95) in class 0: it keeps the first two candidates by index instead of the two most confident.
* **E** drops index 1 and index 7: it uses a different (higher) threshold, e.g. $0.9$ — index 1 (0.85) and index 7 (0.80) are both valid candidates at $\tau=0.80$.

In [ ]:
import torch

PROBS = torch.tensor([[0.91, 0.05, 0.04], [0.10, 0.85, 0.05], [0.82, 0.10, 0.08], [0.05, 0.05, 0.90],
                      [0.95, 0.03, 0.02], [0.30, 0.65, 0.05], [0.80, 0.15, 0.05], [0.06, 0.14, 0.80]],
                     dtype=torch.float64)


def select(probs, tau, cap, strict=False, use_cap=True, by_index=False):
    conf, lab = probs.max(dim=1)
    keep = []
    for k in range(probs.shape[1]):
        cand = [i for i in range(len(conf)) if lab[i] == k and (conf[i] > tau if strict else conf[i] >= tau)]
        if not by_index:
            cand.sort(key=lambda i: (-conf[i].item(), i))
        keep += cand[:cap] if use_cap else cand
    keep.sort()
    return keep, [int(lab[i]) for i in keep]


idx, labels = select(PROBS, 0.80, 2)
print("correct:", idx, labels)
print("strict >:", select(PROBS, 0.80, 2, strict=True))
print("no cap:", select(PROBS, 0.80, 2, use_cap=False))
print("first-by-index:", select(PROBS, 0.80, 2, by_index=True))
print("tau 0.9:", select(PROBS, 0.90, 2))
ANSWER = "B"

### Answer check

In [ ]:
assert (idx, labels) == ([0, 1, 3, 4, 7], [0, 1, 2, 0, 2])
assert select(PROBS, 0.80, 2, strict=True) == ([0, 1, 3, 4], [0, 1, 2, 0])                 # option A
assert select(PROBS, 0.80, 2, use_cap=False) == ([0, 1, 2, 3, 4, 6, 7], [0, 1, 0, 2, 0, 0, 2])  # option C
assert select(PROBS, 0.80, 2, by_index=True) == ([0, 1, 2, 3, 7], [0, 1, 0, 2, 2])          # option D
assert select(PROBS, 0.90, 2) == ([0, 3, 4], [0, 2, 0])                                       # option E
assert ANSWER == "B"